**Set environment**

In [1]:
import numpy  as np
import pandas as pd
import itertools as it
import os

from Bio import SeqIO

In [2]:
%run ../run_config_project.py
show_env()

BASE DIRECTORY (FD_BASE): /hpc/group/igvf/kk319
REPO DIRECTORY (FD_REPO): /hpc/group/igvf/kk319/repo
WORK DIRECTORY (FD_WORK): /hpc/group/igvf/kk319/work
DATA DIRECTORY (FD_DATA): /hpc/group/igvf/kk319/data


You are working with      IGVF BlueSTARR
PATH OF PROJECT (FD_PRJ): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR
PROJECT RESULTS (FD_RES): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results
PROJECT SCRIPTS (FD_EXE): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PROJECT DATA    (FD_DAT): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/data
PROJECT NOTE    (FD_NBK): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/notebooks
PROJECT DOCS    (FD_DOC): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/docs
PROJECT LOG     (FD_LOG): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log
PROJECT REF     (FD_REF): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/references



## Import data

**Check file existent**

In [3]:
%env FD_RES={FD_RES}

env: FD_RES=/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results


In [4]:
%%bash
FDIRY=${FD_RES}/analysis_variant_motif_richard
ls ${FDIRY}/variant_closed_gof_bluestarr_flank35*.fa | xargs -n 1 basename

variant_closed_gof_bluestarr_flank35_obs.fa
variant_closed_gof_bluestarr_flank35_ref.fa
variant_closed_gof_bluestarr_flank35_unobs.fa


In [5]:
%%bash
FDIRY=${FD_RES}/analysis_variant_motif_richard
ls ${FDIRY}/variant_closed_gof_bluestarr_flankL35R70*.fa | xargs -n 1 basename

variant_closed_gof_bluestarr_flankL35R70_obs.fa
variant_closed_gof_bluestarr_flankL35R70_ref.fa
variant_closed_gof_bluestarr_flankL35R70_unobs.fa


**Import data**

In [6]:
### set file directory
txt_fdiry = os.path.join(FD_RES, "analysis_variant_motif_richard")

#txt_fpath_ref   = os.path.join(txt_fdiry, "variant_closed_gof_bluestarr_flank35_ref.fa")
#txt_fpath_obs   = os.path.join(txt_fdiry, "variant_closed_gof_bluestarr_flank35_obs.fa")
#txt_fpath_unobs = os.path.join(txt_fdiry, "variant_closed_gof_bluestarr_flank35_unobs.fa")
txt_fpath_ref = os.path.join(txt_fdiry, "variant_closed_gof_bluestarr_flankL35R70_ref.fa")
txt_fpath_obs = os.path.join(txt_fdiry, "variant_closed_gof_bluestarr_flankL35R70_obs.fa")
txt_fpath_ubs = os.path.join(txt_fdiry, "variant_closed_gof_bluestarr_flankL35R70_unobs.fa")

### import data
lst_seq_ref = list(SeqIO.parse(txt_fpath_ref, "fasta"))
lst_seq_obs = list(SeqIO.parse(txt_fpath_obs, "fasta"))
lst_seq_ubs = list(SeqIO.parse(txt_fpath_ubs, "fasta"))

**check imported data**

In [7]:
print(type(lst_seq_ref[0]))

<class 'Bio.SeqRecord.SeqRecord'>


In [9]:
print(lst_seq_ref[0])

ID: chr4:74487586:A:G:C
Name: chr4:74487586:A:G:C
Description: chr4:74487586:A:G:C
Number of features: 0
Seq('ACATGTGTGTATGCTTGTCTATATACACTGTATGAAGCAAGAGGAGAAAATACT...TTG')


In [7]:
print(lst_seq_ref[0].seq[34:37])

AAG


In [8]:
print(lst_seq_obs[0].seq[34:37])

AGG


In [10]:
print(lst_seq_ubs[0].seq[34:37])

ACG


In [11]:
### check data
assert len(lst_seq_ref) == len(lst_seq_obs) == len(lst_seq_ubs)
num_total = len(lst_seq_ref)
print("Total variants:", num_total)

Total variants: 1884977


## Create pilot (larger chunk) batches for each fasta

In [12]:
def chunked(iterable, size):
    lst = iter(iterable)
    while True:
        chunk = list(it.islice(lst, size))
        if not chunk:
            break
        yield chunk

In [13]:
### output folder for dev batches
txt_fdiry_batch = os.path.join(FD_RES, "analysis_variant_motif_richard", "batches_top01k")
os.makedirs(txt_fdiry_batch, exist_ok=True)

### define batch sizes
num_batch_size = 1_000
num_chunk_size = 1_000

### define file name prefix
txt_prefix = "variant_closed_gof_bluestarr_flankL35R70"

### get the whole batches
lst_batch_ref = lst_seq_ref[:num_batch_size]
lst_batch_obs = lst_seq_obs[:num_batch_size]
lst_batch_ubs = lst_seq_ubs[:num_batch_size]
lst_batch_idx = [rec.id for rec in lst_batch_ref]

### zipped aligned triplets with sequence index
lst_batch_all = zip(lst_batch_idx, lst_batch_ref, lst_batch_obs, lst_batch_ubs)

### init: variant-to-chunk index table
lst_chunk_row = []

### split by chunks (using the chunked function)
idx_chunk = 1
for lst_chunk_all in chunked(lst_batch_all, num_chunk_size):

    ### unpack chunk into lists
    lst_chunk_ref   = [r for (i, r, o, u) in lst_chunk_all]
    lst_chunk_obs   = [o for (i, r, o, u) in lst_chunk_all]
    lst_chunk_ubs   = [u for (i, r, o, u) in lst_chunk_all]
    lst_chunk_idx   = [i for (i, r, o, u) in lst_chunk_all]

    ### set file names
    txt_fpath_out_ref = os.path.join(txt_fdiry_batch, f"{txt_prefix}_pilot_chunk{idx_chunk:03d}_ref.fa")
    txt_fpath_out_obs = os.path.join(txt_fdiry_batch, f"{txt_prefix}_pilot_chunk{idx_chunk:03d}_obs.fa")
    txt_fpath_out_ubs = os.path.join(txt_fdiry_batch, f"{txt_prefix}_pilot_chunk{idx_chunk:03d}_unobs.fa")
    print(f"Chunk {idx_chunk}: {len(lst_chunk_ref)} sequences")

    ### write FASTA files (using SeqIO or your own code)
    SeqIO.write(lst_chunk_ref, txt_fpath_out_ref, "fasta")
    SeqIO.write(lst_chunk_obs, txt_fpath_out_obs, "fasta")
    SeqIO.write(lst_chunk_ubs, txt_fpath_out_ubs, "fasta")
    print(f"Wrote pilot batch of sequences {txt_prefix}_pilot_chunk{idx_chunk:03d}_ref.fa to {txt_fdiry_batch}\n")

    ### add rows to index table
    for idx_variant in lst_chunk_idx:
        lst_chunk_row.append({"Variant_idx": idx_variant, "Chunk_idx": idx_chunk})

    ### increment loop index
    idx_chunk += 1

Chunk 1: 1000 sequences
Wrote pilot batch of sequences variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_ref.fa to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/batches_top01k



In [17]:
dat_chunk_row = pd.DataFrame(lst_chunk_row)

txt_fname_out_idx = f"{txt_prefix}_pilot_chunk_index.tsv"
txt_fpath_out_idx = os.path.join(
    txt_fdiry_batch,
    txt_fname_out_idx
)

dat_chunk_row.to_csv(txt_fpath_out_idx, sep="\t", index=False)
print("Saved:", txt_fname_out_idx)

print(dat_chunk_row.head)

Saved: variant_closed_gof_bluestarr_flankL35R70_pilot_chunk_index.tsv
<bound method NDFrame.head of                  Variant_idx  Chunk_idx
0        chr4:74487586:A:G:C          1
1       chr9:135685432:G:G:C          1
2       chr1:173251490:G:G:C          1
3       chr1:147523391:G:G:C          1
4       chr18:54580749:A:G:T          1
...                      ...        ...
99995   chr5:179109978:T:T:G         20
99996  chr11:133620873:A:A:T         20
99997  chr12:129989192:A:T:C         20
99998    chr7:52991571:A:G:T         20
99999    chr8:72947127:A:T:G         20

[100000 rows x 2 columns]>
